# 02 — Figure per la tesi

Esporta le figure in **PNG e PDF**: il PDF è vettoriale e in LaTeX resta nitido
a qualsiasi zoom. Le figure finiscono nella cartella datata del report, accanto
al markdown che le referenzia con path relativi, così l'insieme resta
autoconsistente e spostabile.


In [ ]:
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
df = pd.read_parquet(ROOT / "data.parquet")
ok = df[df.status == "ok"].copy()

OUT = ROOT / "reports" / datetime.now().strftime("%Y-%m-%d_%H-%M-%S") / "figures"
OUT.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 120, "savefig.bbox": "tight",
    "font.size": 10, "axes.grid": True, "grid.alpha": 0.3,
})

def save(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(OUT / f"{name}.{ext}")
    print(OUT / f"{name}.pdf")


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ok.pivot_table(index="backend", columns="quantization",
               values="lat_median_ms", aggfunc="median").plot.bar(ax=ax)
ax.set_ylabel("latenza mediana [ms]"); ax.set_xlabel("")
save(fig, "latency_by_backend")


In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4.5))
pts = ok.dropna(subset=["lat_median_ms", "acc_map50"])
for key, grp in pts.groupby("backend"):
    ax.scatter(grp.lat_median_ms, grp.acc_map50, label=key, s=30)
ax.set_xlabel("latenza mediana [ms]"); ax.set_ylabel("mAP@50"); ax.legend()
save(fig, "pareto")


In [ ]:
if {"freq_target", "lat_median_ms"} <= set(ok.columns):
    fig, ax = plt.subplots(figsize=(7, 4))
    ok.pivot_table(index="freq_target", columns="compute_target",
                   values="lat_median_ms", aggfunc="median").plot.bar(ax=ax)
    ax.set_ylabel("latenza mediana [ms]"); ax.set_xlabel("profilo di potenza")
    save(fig, "latency_by_profile")
